# Методы классификации: kNN, наивный Байес, деревья и ансамбли

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные и протокол сравнения моделей
2. Метод ближайших соседей (kNN)
3. Наивный байесовский классификатор
4. Деревья решений и ансамбли
5. Сравнение моделей
6. Практикум

На третьем занятии мы разобрали логистическую регрессию и метрики качества. Теперь познакомимся с другими семействами моделей классификации и научимся **честно их сравнивать**:

- **kNN** — «ленивое» обучение на расстояниях;
- **наивный Байес** — вероятностная модель на основе теоремы Байеса;
- **деревья и ансамбли** — нелинейные модели и усреднение/бустинг.

Задача та же — прогноз оттока клиентов, поэтому все модели сравниваются на одной схеме валидации. Практикум (раздел 6) выполняется на демо-датасете этого занятия. Запуски логируем в W&B. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

Среда та же, что и в третьем занятии. **Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")`.

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score, accuracy_score, confusion_matrix,
)

import wandb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42

wandb.login()

## 1. Данные и протокол сравнения моделей

Датасет и целевая переменная те же, что в третьем занятии, — Telco Customer Churn (прогноз ухода клиента). Повторять аудит не нужно.

**Главная мысль раздела:** разные модели требуют разного препроцессинга.

- **kNN и логистическая регрессия** — чувствительны к масштабу: kNN считает расстояния, логистическая регрессия штрафует веса. Признаки нужно стандартизовать.
- **Деревья и ансамбли** — сравнивают пороги значений, масштаб не важен.
- **Наивный Байес** — зависит от варианта модели (у нас — `GaussianNB` и `BernoulliNB`).

Поэтому препроцессинг оформляем функциями и всегда держим его **внутри `Pipeline`**, чтобы статистики не утекали из validation и test. Основная метрика сравнения — **Average Precision (AP)**, дополнительно ROC-AUC и F1 при пороге 0.5.

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
            "master/data/Telco-Customer-Churn.csv")
LOCAL_PATH = "telco_churn.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    df = pd.read_csv(DATA_URL)
    df.to_csv(LOCAL_PATH, index=False)

d = df.drop(columns=['customerID']).copy()
d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce')
d['Churn'] = (d['Churn'] == 'Yes').astype(int)

num = [c for c in d.select_dtypes(include=np.number).columns if c != 'Churn']
cat = [c for c in d.columns if c not in num + ['Churn']]

X, y = d[num + cat], d['Churn']
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE
)

print("Размер таблицы:", d.shape, "| доля оттока:", round(y.mean(), 3))
print("Train / val / test:", len(X_train), len(X_val), len(X_test))


def make_pipeline(model, scale=False):
    """Свежий препроцессор под каждую модель: пропуски + кодирование (+ масштаб при scale)."""
    if scale:
        num_block = Pipeline([('imp', SimpleImputer(strategy='median')),
                              ('sc', StandardScaler())])
    else:
        num_block = SimpleImputer(strategy='median')
    prep = ColumnTransformer([
        ('num', num_block, num),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat),
    ])
    return Pipeline([('prep', prep), ('clf', model)])


def val_metrics(model):
    """AP, ROC-AUC и F1 на validation — модели уже обучены."""
    p = model.predict_proba(X_val)[:, 1]
    return (average_precision_score(y_val, p),
            roc_auc_score(y_val, p),
            f1_score(y_val, (p >= 0.5).astype(int)))

## 2. Метод ближайших соседей (kNN)

### Математический аппарат

Обучения как такового нет: для нового объекта находим $k$ ближайших соседей в обучающей выборке и берём мажоритарный класс (или среднюю вероятность). Расстояние — обычно евклидово или манхэттенское:

$$d(a, b) = \sqrt{\sum_{j=1}^{d}(a_j - b_j)^2}$$

Число соседей $k$ — главный гиперпараметр и регуляризатор:

- **малое $k$** (например, 1) — модель почти запоминает выборку: низкий bias, огромная дисперсия, переобучение;
- **большое $k$** — усреднение по многим соседям: сглаживает шум, но может недообучиться.

*Смысл:* kNN опирается на **метрику близости**, поэтому масштаб признаков критичен (рубль и год не сравнимы напрямую), а в больших размерностях расстояния становятся почти одинаковыми — это **проклятие размерности**.

**Задача 1.**
Постройте график val AP в зависимости от $k$ (1, 3, 5, 11, 25, 50). Где модель переобучается, а где начинает недообучаться? Объясните через bias–variance. Ответ — в «Выводе».

In [ ]:
ks = [1, 3, 5, 11, 25, 50]
rows = []
for k in ks:
    m = make_pipeline(KNeighborsClassifier(n_neighbors=k), scale=True).fit(X_train, y_train)
    ap, roc, f1 = val_metrics(m)
    rows.append((k, ap, roc))

knn_table = pd.DataFrame(rows, columns=['k', 'val_AP', 'val_ROC_AUC'])
print(knn_table.round(3).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(knn_table['k'], knn_table['val_AP'], marker='o')
plt.xscale('log')
plt.xticks(knn_table['k'], knn_table['k'])
plt.xlabel('число соседей k')
plt.ylabel('validation AP')
plt.title('kNN: качество против числа соседей')
plt.show()

**Вывод по kNN:**

_..._

## 3. Наивный байесовский классификатор

### Математический аппарат

Модель применяет теорему Байеса к признакам объекта:

$$P(y \mid x) = \frac{P(y)\,P(x \mid y)}{P(x)} \propto P(y)\prod_{j=1}^{d} P(x_j \mid y)$$

Ключевое (и «наивное») допущение — **условная независимость признаков** при известном классе: совместная вероятность распадается в произведение одномерных. Это почти всегда неверно, но модель на удивление работоспособна.

Варианты зависят от типа признаков:

- **GaussianNB** — для непрерывных признаков (каждый описывается нормальным распределением);
- **MultinomialNB** — для счётчиков (текст «мешок слов»);
- **BernoulliNB** — для бинарных/one-hot признаков.

*Смысл:* наивный Байес очень быстрый, почти без гиперпараметров, и служит отличным baseline. Его вероятности плохо калиброваны (связь с P.S. третьего занятия), но **ранжирование** часто оказывается неплохим.

**Задача 2.**
Сравните `GaussianNB` и `BernoulliNB` по val AP. Объясните разницу через допущения каждой модели о признаках. Ответ — в «Выводе».

In [ ]:
gnb = make_pipeline(GaussianNB()).fit(X_train, y_train)
bnb = make_pipeline(BernoulliNB()).fit(X_train, y_train)

nb_table = pd.DataFrame([
    ('GaussianNB',) + val_metrics(gnb),
    ('BernoulliNB',) + val_metrics(bnb),
], columns=['model', 'val_AP', 'val_ROC_AUC', 'val_F1@0.5'])
print(nb_table.round(3).to_string(index=False))

**Примечание.** `BernoulliNB` считает признаки бинарными (порог по умолчанию — 0). Числовые признаки вроде `tenure` или `MonthlyCharges` при этом теряют информацию о величине, поэтому на смешанных данных `GaussianNB` обычно уместнее. Для текстов, наоборот, `BernoulliNB` и `MultinomialNB` — основной выбор.

**Вывод по наивному Байесу:**

_..._

## 4. Деревья решений и ансамбли

### Математический аппарат

Дерево рекурсивно разбивает пространство признаков, измеряя однородность узла **нечистотой**:

$$Gini = 1 - \sum_{k} p_k^2, \qquad Entropy = -\sum_{k} p_k \log p_k$$

Прирост информации при разбиении — уменьшение нечистоты: $IG = H(\text{родитель}) - \sum_j \frac{n_j}{n} H(\text{ребёнок}_j)$. Дерево без ограничений запоминает выборку (train AP → 1.0), поэтому глубину ограничивают.

**Ансамбли** уменьшают разные части ошибки (bias–variance):

- **случайный лес** — бэггинг с усреднением; дисперсия среднего $B$ моделей с корреляцией $\rho$ равна $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$, поэтому лес ещё и уменьшает корреляцию, выбирая случайные подмножества признаков;
- **градиентный бустинг** — деревья строятся последовательно, каждое исправляет ошибки предыдущих; это градиентный спуск в функциональном пространстве, и он уменьшает в первую очередь **bias**.

**Задача со звёздочкой.**
Выведите формулу дисперсии среднего ансамбля и объясните, почему случайный лес выигрывает у простого усреднения одинаковых деревьев.

**Ваше доказательство:**

_..._

In [ ]:
# Переобучение дерева: качество на train и на validation
rows = []
for depth in [2, 3, 5, 8, None]:
    m = make_pipeline(DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)).fit(X_train, y_train)
    rows.append((str(depth),
                 average_precision_score(y_train, m.predict_proba(X_train)[:, 1]),
                 average_precision_score(y_val, m.predict_proba(X_val)[:, 1])))

depth_table = pd.DataFrame(rows, columns=['max_depth', 'train_AP', 'val_AP'])
depth_table['x'] = range(len(depth_table))
print(depth_table[['max_depth', 'train_AP', 'val_AP']].round(3).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(depth_table['x'], depth_table['train_AP'], marker='o', label='train AP')
plt.plot(depth_table['x'], depth_table['val_AP'], marker='o', label='validation AP')
plt.xticks(depth_table['x'], depth_table['max_depth'])
plt.xlabel('max_depth')
plt.ylabel('Average Precision')
plt.title('Переобучение дерева решений')
plt.legend()
plt.show()

In [ ]:
rf = make_pipeline(RandomForestClassifier(
    n_estimators=300, max_depth=8, n_jobs=-1, random_state=RANDOM_STATE)).fit(X_train, y_train)
gb = make_pipeline(HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.1, random_state=RANDOM_STATE)).fit(X_train, y_train)
tree3 = make_pipeline(DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)).fit(X_train, y_train)

for name, m in [('Tree depth=3', tree3), ('RandomForest', rf), ('HistGradientBoosting', gb)]:
    ap, roc, f1 = val_metrics(m)
    print(f"{name:22s} val AP = {ap:.3f}, ROC-AUC = {roc:.3f}, F1@0.5 = {f1:.3f}")

**Вывод по деревьям и ансамблям:**

_..._

### Важность признаков

У линейных моделей интерпретация — коэффициенты. У деревьев есть **impurity importance** (уменьшение нечистоты; считается по обучающей выборке и смещена к признакам с большим числом значений) и **permutation importance** (падение качества на validation при перемешивании признака; честнее, но дороже и размывается при коррелирующих признаках).

In [ ]:
# Impurity importance случайного леса
names = rf.named_steps['prep'].get_feature_names_out()
impurity = pd.Series(rf.named_steps['clf'].feature_importances_, index=names).sort_values(ascending=False)
print("Топ-10 по impurity importance (RandomForest):")
print(impurity.head(10).round(3).to_string())

plt.figure(figsize=(8, 5))
impurity.head(10).sort_values().plot(kind='barh')
plt.xlabel('Важность')
plt.title('Топ-10 признаков (impurity importance)')
plt.show()

In [ ]:
# Permutation importance считается по исходным колонкам
perm = permutation_importance(
    gb, X_val, y_val, n_repeats=5, random_state=RANDOM_STATE,
    scoring='average_precision',
)
perm_series = pd.Series(perm.importances_mean, index=X_val.columns).sort_values(ascending=False)
print("Топ-10 по permutation importance (HistGradientBoosting):")
print(perm_series.head(10).round(4).to_string())

## 5. Сравнение моделей

Собираем все модели в одну таблицу на **одной валидации**, выбираем лучшую по val AP и один раз оцениваем на test.

Логистическая регрессия из третьего занятия здесь — точка отсчёта: интересно, обыграют ли её более сложные модели.

In [ ]:
logreg = make_pipeline(
    LogisticRegression(C=0.3, max_iter=1500, solver='liblinear'), scale=True
).fit(X_train, y_train)

best_k = int(knn_table.loc[knn_table['val_AP'].idxmax(), 'k'])
knn_best = make_pipeline(KNeighborsClassifier(n_neighbors=best_k), scale=True).fit(X_train, y_train)

models = {
    'LogReg (занятие 3)': logreg,
    f'kNN (k={best_k})': knn_best,
    'GaussianNB': gnb,
    'BernoulliNB': bnb,
    'Tree depth=3': tree3,
    'RandomForest': rf,
    'HistGradientBoosting': gb,
}

rows = []
for name, m in models.items():
    ap, roc, f1 = val_metrics(m)
    rows.append((name, ap, roc, f1))

cmp_table = pd.DataFrame(rows, columns=['model', 'val_AP', 'val_ROC_AUC', 'val_F1@0.5'])
cmp_table = cmp_table.sort_values('val_AP', ascending=False).reset_index(drop=True)
print(cmp_table.round(3).to_string(index=False))

In [ ]:
# Лучшая по валидации модель — финальная оценка на test (один раз)
best_name = cmp_table.iloc[0]['model']
best_model = models[best_name]

p_test = best_model.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.5).astype(int)

result = {
    'best_model': best_name,
    'val_AP': round(float(cmp_table.iloc[0]['val_AP']), 3),
    'test_AP': average_precision_score(y_test, p_test),
    'test_ROC_AUC': roc_auc_score(y_test, p_test),
    'test_F1@0.5': f1_score(y_test, pred_test),
    'test_accuracy': accuracy_score(y_test, pred_test),
}
print("Confusion matrix на test:")
print(confusion_matrix(y_test, pred_test))
for k, v in result.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

run = wandb.init(project="apml-course", name="classification-methods-churn",
                 config={"models": list(models), "best_by": "val_AP"})
wandb.log({k: v for k, v in result.items() if isinstance(v, (int, float))})
run.finish()

**Примечание.** Более сложная модель не обязана выигрывать. На этом датасете логистическая регрессия оказывается на уровне лучших ансамблей, а kNN и наивный Байес отстают, но остаются полезными как быстрые baseline. Вывод для практики: сравнивайте модели честно — на одной валидации и по заранее выбранной метрике.

**Задача 3.**
Назовите лучшую модель и модели-аутсайдеры. Если разрыв между лучшей и второй мал, что это значит для выбора модели с точки зрения интерпретируемости, скорости и стоимости поддержки? Ответ — в «Вашем ответе».

**Ваш ответ:**

_..._

## 6. Практикум

Задания выполняются на **датасете этого занятия** (Telco Customer Churn). Каждый запуск логируйте в W&B отдельным run. После каждого задания заполните ячейку **«Вывод»**.

### Задание 6.1. kNN

Заново переберите число соседей $k$ (например, 1, 3, 5, 11, 25, 50, 100). Покажите переобучение при малых $k$ и выберите оптимальное значение по validation. Проверьте, что без масштабирования признаков качество падает.

**Вывод:**

_..._

### Задание 6.2. Наивный Байес

Сравните `GaussianNB` и `BernoulliNB` по val AP, ROC-AUC и F1. Объясните, какой вариант лучше подходит к смешанным (числовым и категориальным) признакам и почему.

**Вывод:**

_..._

### Задание 6.3. Дерево решений

Переберите `max_depth` и постройте график train AP против val AP. Покажите точку, после которой начинается переобучение.

**Вывод:**

_..._

### Задание 6.4. Ансамбли

Обучите `RandomForestClassifier` и `HistGradientBoostingClassifier`. Сравните их с одиночным деревом и с логистической регрессией по val AP, ROC-AUC и F1.

**Вывод:**

_..._

### Задание 6.5. Важность признаков

Для случайного леса выведите топ-10 по impurity importance, для бустинга — топ-10 по permutation importance. Сопоставьте их с коэффициентами логистической регрессии из третьего занятия: какие признаки устойчиво важны во всех трёх подходах?

**Вывод:**

_..._

### Задание 6.6. Итоговое резюме

Соберите все модели в одну таблицу на одной валидации, выберите лучшую по val AP, один раз оцените на test и залогируйте в W&B. В 5–7 предложениях: какая модель победила и почему; совпало ли это с ожиданием; какие признаки оказались важными; какую модель вы бы взяли в продакшн и при каких условиях.

**Резюме:**

_..._

---

## P.S. Почему наивный Байес работает при нарушении независимости

Допущение об условной независимости признаков почти всегда неверно, и вероятности `predict_proba` у наивного Байеса плохо откалиброваны (связь с P.S. третьего занятия). Тем не менее **классификация** часто выходит неплохой.

Причина в том, что для решения нужен не точный $P(y \mid x)$, а лишь **правильный знак сравнения** вероятностей классов:

$$\arg\max_y P(y)\prod_j P(x_j \mid y)$$

Умножение на лишние сомножители искажает вероятности, но может сохранять их **порядок** относительно классов. Поэтому наивный Байес нередко даёт приемлемое ранжирование (AUC) при плохой калибровке. Это ещё раз показывает: метрику и порог надо выбирать под задачу, а «красивую» вероятность проверять отдельно.

### Справочные материалы

- kNN (scikit-learn): https://scikit-learn.org/stable/modules/neighbors.html#classification
- Наивный Байес: https://scikit-learn.org/stable/modules/naive_bayes.html
- Решающие деревья: https://scikit-learn.org/stable/modules/tree.html
- Ансамбли: https://scikit-learn.org/stable/modules/ensemble.html
- Важность признаков: https://scikit-learn.org/stable/modules/permutation_importance.html